# Спринт 25. Классификация транзакций TechGadget и трекинг экспериментов в MLflow

**Задача.** Построить модель, выявляющую мошеннические транзакции (`fraud_flag = 1`), и зафиксировать весь процесс в MLflow: параметры, метрики, графики, датасет, модель, версию в Model Registry.

**Структура тетради**
1. Чтение данных
2. Исследовательский анализ и проверка гипотез
3. Предобработка
4. Подключение к MLflow
5. Обучение `XGBClassifier`: случайный поиск гиперпараметров, один набор = один run
6. Выбор лучшей модели, регистрация в Model Registry, контроль на тесте
7. Автопроверка `validate_mlflow.py`, фиксация окружения
8. Итоговый вывод

**Ограничения валидатора, учтённые в коде**
- проверяются **все** запуски эксперимента → в эксперименте лежат только обучающие run'ы, EDA-графики прикладываются к каждому из них в папку `eda/`;
- имена метрик — только из списка `accuracy, precision, recall, f1, f1_score, roc_auc, roc-auc` → время обучения и тестовые метрики пишутся **тегами**, `autolog` не используется;
- наборы гиперпараметров не должны повторяться → при повторном запуске тетради уже залогированные комбинации пропускаются.

In [ ]:
import os
import sys
import time
import pickle
import warnings
import subprocess
import urllib.request
from pathlib import Path
from importlib import metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
)

import shap
import xgboost as xgb
from xgboost import XGBClassifier

import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("mlflow :", mlflow.__version__, "(на сервере 3.6.0 — версии должны совпадать)")
print("xgboost:", xgb.__version__, "| shap:", shap.__version__)

In [ ]:
# ---- Конфигурация проекта ----
RANDOM_STATE = 42
TEST_SIZE = 0.20      # отложенный тест
VALID_SIZE = 0.25     # доля валидации внутри train (0.25 * 0.8 = 0.2 от всех данных)
N_RUNS = 6            # требование: минимум 3
ALPHA = 0.05          # уровень значимости для хи-квадрат

EXPERIMENT_NAME = "fraud_detection_classification"
MODEL_NAME = "FraudDetectionModel"
TARGET = "fraud_flag"

DATA_FILE = Path("retail_fraud_detection_100k.csv")
DATA_URL = "https://code.s3.yandex.net/datasets/retail_fraud_detection_100k.csv"

ARTIFACT_DIR = Path("artifacts")
EDA_DIR = ARTIFACT_DIR / "eda"
RUNS_DIR = ARTIFACT_DIR / "runs"
for d in (EDA_DIR, RUNS_DIR):
    d.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"


def save_fig(fig, path):
    # Сохраняет график на диск (для MLflow), показывает в тетради и закрывает.
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Шаг 1. Чтение данных

In [ ]:
df = pd.read_csv(DATA_FILE if DATA_FILE.exists() else DATA_URL)
print("Размер:", df.shape)
display(df.head())
df.info()

## Шаг 2. Анализ и исследование данных
### 2.1. Статистика, пропуски, дубликаты

In [ ]:
display(df.describe().T)
display(df.describe(include="object").T)

In [ ]:
quality = pd.Series({
    "пропусков всего": int(df.isna().sum().sum()),
    "полных дубликатов строк": int(df.duplicated().sum()),
    "дубликатов transaction_id": int(df["transaction_id"].duplicated().sum()),
})
display(quality.to_frame("значение"))
print(f"Доля фрода: {df[TARGET].mean():.3f}")
display(pd.crosstab(df["fraud_risk"], df[TARGET], margins=True))

**Наблюдения**
- Пропусков и дубликатов нет — заполнение/удаление не требуется.
- Классы почти сбалансированы (≈47.5 % фрода), поэтому `scale_pos_weight` и ресэмплинг не нужны, а accuracy остаётся информативной метрикой.
- **`fraud_risk` — утечка целевой переменной:** `High` → 100 % фрода, `Low` → 0 %. Признак производен от таргета и в момент транзакции недоступен, в модель не идёт.

### 2.2. Признаки из `transaction_timestamp`

In [ ]:
df["transaction_timestamp"] = pd.to_datetime(df["transaction_timestamp"])
df["hour"] = df["transaction_timestamp"].dt.hour
df["day_of_week"] = df["transaction_timestamp"].dt.dayofweek   # 0 = понедельник

print("Период:", df["transaction_timestamp"].min(), "—", df["transaction_timestamp"].max())
print("Уникальных значений hour:", df["hour"].nunique(), "| day_of_week:", df["day_of_week"].nunique())

`hour` принимает единственное значение (датасет синтетический, все метки времени сгенерированы в одно и то же время суток) — признак **константный** и на шаге предобработки будет отброшен автоматически. `day_of_week` информативен по структуре, его связь с фродом проверяется в гипотезе 3.

### 2.3. Распределения и выбросы

In [ ]:
CONT_COLS = ["transaction_amount", "avg_transaction_amount_7d", "transaction_frequency_24h",
             "failed_transaction_count_24h", "account_age_days", "day_of_week"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), CONT_COLS):
    discrete = df[col].nunique() < 30
    sns.histplot(data=df, x=col, hue=TARGET, ax=ax, stat="density", common_norm=False,
                 discrete=discrete, bins=None if discrete else 40, alpha=0.5)
    ax.set_title(col)
fig.suptitle("Распределения числовых признаков в разрезе fraud_flag", y=1.01)
fig.tight_layout()
save_fig(fig, EDA_DIR / "feature_distributions.png")

In [ ]:
OUTLIER_COLS = ["transaction_amount", "avg_transaction_amount_7d", "transaction_frequency_24h",
                "failed_transaction_count_24h", "account_age_days"]
rows = []
for col in OUTLIER_COLS:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    mask = (df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)
    rows.append({"признак": col, "выбросов": int(mask.sum()), "доля, %": round(mask.mean() * 100, 2),
                 "доля фрода среди выбросов": round(df.loc[mask, TARGET].mean(), 3) if mask.any() else np.nan})
display(pd.DataFrame(rows))

fig, axes = plt.subplots(1, len(OUTLIER_COLS), figsize=(18, 4))
for ax, col in zip(axes, OUTLIER_COLS):
    sns.boxplot(data=df, y=col, x=TARGET, ax=ax)
    ax.set_title(col, fontsize=10)
fig.suptitle("Выбросы (правило 1.5·IQR)", y=1.03)
fig.tight_layout()
save_fig(fig, EDA_DIR / "outliers_boxplot.png")

**Выбросы** есть только в `transaction_amount` (длинный правый хвост), и доля фрода среди них заметно выше средней — это сигнал, а не шум. Выбросы **не удаляются**: нетипичные суммы — один из признаков мошенничества, а деревья решений к масштабу и хвостам нечувствительны.

### 2.4. Где фрод встречается чаще: страны, категории, устройства

In [ ]:
CAT_COLS = ["location", "merchant_category", "device_type", "payment_method"]
base_rate = df[TARGET].mean()

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, col in zip(axes, CAT_COLS):
    rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    sns.barplot(x=rate.values, y=rate.index, ax=ax, color="steelblue")
    ax.axvline(base_rate, color="red", ls="--", lw=1, label="средняя доля")
    ax.set_xlim(rate.min() - 0.05, rate.max() + 0.03)
    ax.set_title(f"Доля фрода: {col}"); ax.set_xlabel(""); ax.set_ylabel("")
axes[0].legend()
fig.tight_layout()
save_fig(fig, EDA_DIR / "fraud_rate_by_category.png")

### 2.5. Корреляции

In [ ]:
num_df = df.select_dtypes("number").drop(columns=["hour"])   # hour константен → корреляция не определена
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, annot_kws={"size": 8})
ax.set_title("Корреляционная матрица признаков (Пирсон)")
save_fig(fig, EDA_DIR / "correlation_matrix.png")

display(corr[TARGET].drop(TARGET).sort_values(ascending=False).to_frame("corr c fraud_flag"))
print("is_international == unusual_location_flag во всех строках:",
      bool((df["is_international"] == df["unusual_location_flag"]).all()))

**Наблюдения**
- Сильнее всего с фродом связаны `previous_fraud_flag`, `is_international` / `unusual_location_flag` (≈0.42), затем флаги скорости и нетипичной суммы (≈0.33–0.35) и `transaction_frequency_24h`.
- `unusual_location_flag` **полностью дублирует** `is_international` (корреляция 1.0) — один из пары удаляется.
- `account_age_days` с таргетом не связан; `avg_transaction_amount_7d` связан отрицательно.

### 2.6. Проверка гипотез (χ², α = 0.05)

In [ ]:
def chi2_test(data, feature, target=TARGET, alpha=ALPHA):
    # H0: признак и fraud_flag независимы. Cramér's V — размер эффекта (на 100k строк p-value мало что говорит).
    ct = pd.crosstab(data[feature], data[target])
    chi2, p, dof, _ = chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    return {"признак": feature, "chi2": round(chi2, 1), "dof": dof, "p_value": p,
            "cramers_v": round(v, 3), "H0 отвергнута": p < alpha}


HYPOTHESES = {
    "H1. Фрод чаще встречается среди международных транзакций": "is_international",
    "H2. Фрод связан с определёнными методами оплаты": "payment_method",
    "H3. Фрод чаще происходит в определённые дни недели": "day_of_week",
    "H4. Для фрода чаще используются высокорисковые устройства": "high_risk_device_flag",
}

hyp_df = pd.DataFrame([{"гипотеза": h, **chi2_test(df, f)} for h, f in HYPOTHESES.items()])
hyp_df["вывод"] = np.where(hyp_df["H0 отвергнута"], "подтверждена", "не подтверждена")
hyp_df.to_csv(EDA_DIR / "hypotheses_chi2.csv", index=False)
display(hyp_df)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (h, f), (_, r) in zip(axes, HYPOTHESES.items(), hyp_df.iterrows()):
    rate = df.groupby(f)[TARGET].mean()
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax, color="indianred" if r["H0 отвергнута"] else "grey")
    ax.axhline(base_rate, color="black", ls="--", lw=1)
    ax.set_title(f"{h.split('.')[0]}: {f}\np={r['p_value']:.3g}, V={r['cramers_v']}", fontsize=10)
    ax.set_ylabel("доля фрода"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
save_fig(fig, EDA_DIR / "hypotheses_fraud_rate.png")

**Итоги проверки гипотез**

| Гипотеза | Результат | Что это значит для модели |
|---|---|---|
| H1. Международные транзакции | **подтверждена**, эффект сильный (V ≈ 0.42): доля фрода ≈ 69 % против ≈ 27 % | ключевой признак; повод для усиленной проверки cross-border платежей |
| H2. Методы оплаты | **не подтверждена** (p ≫ 0.05): доли фрода по методам 47–48 % | признак неинформативен; оставлен в модели для выполнения требования по OHE, SHAP подтверждает нулевой вклад |
| H3. Дни недели | **не подтверждена** (p ≫ 0.05): разброс 46.8–48.2 % в пределах шума | недельной сезонности нет |
| H4. Высокорисковые устройства | **подтверждена**, эффект умеренный (V ≈ 0.13): ≈ 66 % против ≈ 45 % | полезный признак |

Дополнительно (п. 2.4): по странам и категориям продавцов доля фрода одинакова (различия статистически незначимы), по типу устройства выделяется **Mobile** (≈ 52 % против ≈ 45 % у Desktop/Tablet).

## Шаг 3. Предобработка

1. Удаляются идентификаторы, исходный timestamp, утечка `fraud_risk`, дубликат `unusual_location_flag` и константные колонки.
2. Разбиение **train / valid / test = 60 / 20 / 20** со стратификацией. Валидация нужна, чтобы выбирать гиперпараметры не по тесту: метрики в run'ах считаются на valid, тест используется один раз — для итоговой проверки зарегистрированной модели.
3. `OneHotEncoder` для категориальных, `StandardScaler` для непрерывных, бинарные флаги не трогаются. Трансформеры обучаются **только на train** (без утечки статистик). Для XGBoost масштабирование не обязательно, но делает препроцессор пригодным для любых моделей.

In [ ]:
DROP_COLS = ["transaction_id", "customer_id", "transaction_timestamp", "fraud_risk"]
if (df["is_international"] == df["unusual_location_flag"]).all():
    DROP_COLS.append("unusual_location_flag")
DROP_COLS += [c for c in df.columns if df[c].nunique() <= 1]          # константы (hour)
print("Удаляем:", DROP_COLS)

X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET]

cat_features = X.select_dtypes("object").columns.tolist()
flag_features = [c for c in X.columns if c not in cat_features and set(X[c].unique()) <= {0, 1}]
num_features = [c for c in X.columns if c not in cat_features + flag_features]
print("Категориальные:", cat_features)
print("Непрерывные   :", num_features)
print("Флаги         :", flag_features)

In [ ]:
X_trainval, X_test_raw, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=VALID_SIZE, stratify=y_trainval, random_state=RANDOM_STATE)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
        ("flag", "passthrough", flag_features),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_train = preprocessor.fit_transform(X_train_raw).astype("float64")
X_valid = preprocessor.transform(X_valid_raw).astype("float64")
X_test = preprocessor.transform(X_test_raw).astype("float64")

for name, part, target in [("train", X_train, y_train), ("valid", X_valid, y_valid), ("test", X_test, y_test)]:
    print(f"{name:5s}: {part.shape}, доля фрода {target.mean():.3f}")

# Артефакты, общие для всех запусков
train_dataset = X_train.assign(**{TARGET: y_train.values})
DATASET_PATH = ARTIFACT_DIR / "train_dataset.parquet"
train_dataset.to_parquet(DATASET_PATH, index=False)

PREPROCESSOR_PATH = ARTIFACT_DIR / "preprocessor.pkl"
with open(PREPROCESSOR_PATH, "wb") as f:
    pickle.dump(preprocessor, f)

display(X_train.head(3))

## Шаг 4. Подключение к MLflow

Данные для подключения вставляются **в одну ячейку ниже** (значения — из выдачи сниппета запуска MLflow). Ячейка сама создаёт файл `.env`, который нужен `validate_mlflow.py`. Если `.env` уже лежит рядом с тетрадью — ячейку можно не трогать, он имеет приоритет.

⚠️ Перед тем как делиться тетрадью или коммитить её, верните в ячейку плейсхолдеры и не добавляйте `.env` в репозиторий.

In [ ]:
# >>> ВСТАВЬТЕ СВОИ ДАННЫЕ ВМЕСТО <...> <<<
CREDS = {
    "MLFLOW_TRACKING_URI": "https://mlflow-ds-<идентификатор>.infra.data-science.education-services.ru",
    "MLFLOW_TRACKING_USERNAME": "<логин>",
    "MLFLOW_TRACKING_PASSWORD": "<пароль>",
    "MLFLOW_S3_ENDPOINT_URL": "https://storage.yandexcloud.net",
    "AWS_ACCESS_KEY_ID": "<access_key_id>",
    "AWS_SECRET_ACCESS_KEY": "<secret_access_key>",
}

ENV_FILE = Path(".env")
creds_filled = not any("<" in v for v in CREDS.values())

if creds_filled:
    # значения из ячейки → в окружение и в .env (его читает validate_mlflow.py)
    CREDS["MLFLOW_TRACKING_URI"] = CREDS["MLFLOW_TRACKING_URI"].rstrip("/")
    if not CREDS["MLFLOW_TRACKING_URI"].startswith("http"):
        CREDS["MLFLOW_TRACKING_URI"] = "https://" + CREDS["MLFLOW_TRACKING_URI"]   # сниппет выдаёт адрес без схемы
    ENV_FILE.write_text("".join(f'{k} = "{v}"\n' for k, v in CREDS.items()))
    print("Данные взяты из ячейки, .env обновлён")
elif ENV_FILE.exists():
    print("В ячейке плейсхолдеры — используется существующий .env")
else:
    raise ValueError("Заполните CREDS в этой ячейке или положите рядом с тетрадью файл .env")

In [ ]:
load_dotenv(override=True)

REQUIRED_ENV = ["MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD",
                "MLFLOW_S3_ENDPOINT_URL", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]
missing = [v for v in REQUIRED_ENV if not os.environ.get(v)]
assert not missing, f"В .env не заданы переменные: {missing}"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
client = MlflowClient()

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    # Имя удалённого эксперимента повторно использовать нельзя: при ошибке здесь поменяйте EXPERIMENT_NAME.
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Эксперимент '{EXPERIMENT_NAME}', id = {experiment_id}")

## Шаг 5. Обучение модели и трекинг экспериментов

Случайный поиск по 6 гиперпараметрам `XGBClassifier`. Диапазоны намеренно широкие (от слабых моделей к сильным), чтобы сравнение запусков было содержательным.

В каждый run пишутся:
- **params** — гиперпараметры + служебные (`random_state`, размеры выборок);
- **metrics** (на valid) — `accuracy`, `precision`, `recall`, `f1`, `roc_auc`;
- **tags** — время обучения и инференса;
- **artifacts** — `model.pkl`, MLflow-модель (`mlflow.xgboost`), `preprocessor.pkl`, `confusion_matrix.png`, `roc_curve.png`, `learning_curve.png`, `shap_summary.png`, EDA-графики (в т. ч. `correlation_matrix.png`, `feature_distributions.png`), обучающий датасет (`parquet` + вкладка Datasets).

In [ ]:
HP_KEYS = ["max_depth", "learning_rate", "n_estimators", "subsample", "colsample_bytree", "min_child_weight"]


def sample_params(rng):
    return {
        "max_depth": int(rng.integers(1, 9)),
        "learning_rate": round(float(10 ** rng.uniform(-2.3, -0.5)), 4),     # лог-равномерно 0.005…0.32
        "n_estimators": int(rng.choice([25, 50, 100, 200, 400])),
        "subsample": round(float(rng.uniform(0.6, 1.0)), 2),
        "colsample_bytree": round(float(rng.uniform(0.6, 1.0)), 2),
        "min_child_weight": int(rng.integers(1, 8)),
    }


def hp_key(params):
    return tuple(str(params[k]) for k in HP_KEYS)


# Комбинации, уже залогированные в эксперименте (защита от дублей при повторном запуске тетради)
existing = mlflow.search_runs(experiment_ids=[experiment_id])
used = set()
if not existing.empty and all(f"params.{k}" in existing.columns for k in HP_KEYS):
    used = {tuple(str(v) for v in row) for row in existing[[f"params.{k}" for k in HP_KEYS]].itertuples(index=False)}

rng = np.random.default_rng(RANDOM_STATE)
param_grid = []
while len(param_grid) < N_RUNS:
    candidate = sample_params(rng)
    if hp_key(candidate) not in used:
        used.add(hp_key(candidate))
        param_grid.append(candidate)

display(pd.DataFrame(param_grid))

In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }


def plot_confusion_matrix(y_true, y_pred, path, title):
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=["legit", "fraud"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title); ax.grid(False)
    save_fig(fig, path)


def plot_roc(y_true, y_proba, path, title):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    fig, ax = plt.subplots(figsize=(5, 4.5))
    ax.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_true, y_proba):.4f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title(title); ax.legend(loc="lower right")
    save_fig(fig, path)


def plot_learning_curve(model, path, title):
    # Кривая обучения: logloss на train/valid по итерациям бустинга.
    history = model.evals_result()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(history["validation_0"]["logloss"], label="train")
    ax.plot(history["validation_1"]["logloss"], label="valid")
    ax.set_xlabel("итерация бустинга"); ax.set_ylabel("logloss"); ax.set_title(title); ax.legend()
    save_fig(fig, path)


def plot_shap(model, X_sample, path, title):
    # TreeSHAP считается самим XGBoost (pred_contribs) — не зависит от совместимости версий shap/xgboost.
    contribs = model.get_booster().predict(xgb.DMatrix(X_sample), pred_contribs=True)
    shap.summary_plot(contribs[:, :-1], X_sample, max_display=15, show=False)   # последний столбец — bias
    fig = plt.gcf()
    fig.suptitle(title, y=1.02)
    save_fig(fig, path)

In [ ]:
def train_and_log(params, run_name):
    run_dir = RUNS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    with mlflow.start_run(run_name=run_name) as run:
        # --- параметры
        mlflow.log_params(params)
        mlflow.log_params({"random_state": RANDOM_STATE, "n_features": X_train.shape[1],
                           "train_rows": len(X_train), "valid_rows": len(X_valid)})
        mlflow.set_tags({"model_type": "XGBClassifier", "metrics_split": "valid", "stage": "random_search"})

        # --- датасет
        mlflow.log_input(mlflow.data.from_pandas(train_dataset, targets=TARGET, name="retail_fraud_train"),
                         context="training")
        mlflow.log_artifact(str(DATASET_PATH), artifact_path="dataset")
        mlflow.log_artifact(str(PREPROCESSOR_PATH), artifact_path="preprocessing")

        # --- обучение
        model = XGBClassifier(**params, objective="binary:logistic", eval_metric="logloss",
                              tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)
        t0 = time.perf_counter()
        model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_valid, y_valid)], verbose=False)
        train_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        y_proba = model.predict_proba(X_valid)[:, 1]
        predict_time = time.perf_counter() - t0
        y_pred = (y_proba >= 0.5).astype(int)

        # --- метрики (только разрешённые валидатором имена), время — тегами
        metrics = compute_metrics(y_valid, y_pred, y_proba)
        mlflow.log_metrics(metrics)
        mlflow.set_tags({"train_time_sec": round(train_time, 3),
                         "predict_ms_per_1k_rows": round(predict_time / len(X_valid) * 1e6, 3)})

        # --- графики
        plot_confusion_matrix(y_valid, y_pred, run_dir / "confusion_matrix.png", f"Confusion matrix — {run_name}")
        plot_roc(y_valid, y_proba, run_dir / "roc_curve.png", f"ROC — {run_name}")
        plot_learning_curve(model, run_dir / "learning_curve.png", f"Learning curve — {run_name}")
        plot_shap(model, X_valid.sample(2000, random_state=RANDOM_STATE), run_dir / "shap_summary.png",
                  f"SHAP — {run_name}")
        mlflow.log_artifacts(str(run_dir), artifact_path="plots")
        mlflow.log_artifacts(str(EDA_DIR), artifact_path="eda")

        # --- модель: pickle + MLflow-формат (нужен для Model Registry)
        model_path = run_dir / "model.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(model, f)
        mlflow.log_artifact(str(model_path), artifact_path="model_pickle")
        model_path.unlink()                         # чтобы pkl не попадал в папку plots при повторных запусках
        model_info = mlflow.xgboost.log_model(model, name="model", signature=infer_signature(X_valid, y_proba),
                                              input_example=X_valid.head(3))
        mlflow.set_tag("model_uri", model_info.model_uri)       # MLflow 3: модель — отдельная сущность LoggedModel

        return {"run_id": run.info.run_id, "run_name": run_name, **params, **metrics,
                "train_time_sec": round(train_time, 3)}


# Нумерация продолжается, если в эксперименте уже есть запуски
start_idx = len(existing) + 1
results = []
for i, params in enumerate(param_grid, start=start_idx):
    run_name = f"xgb_run_{i:02d}"
    print(f"\n===== {run_name}: {params}")
    try:
        results.append(train_and_log(params, run_name))
    except Exception as err:
        # Упавший run остаётся в эксперименте и ломает автопроверку — удаляем его.
        failed = mlflow.search_runs(experiment_ids=[experiment_id], filter_string="attributes.status = 'FAILED'")
        for rid in failed.get("run_id", []):
            client.delete_run(rid)
        raise err

display(pd.DataFrame(results).drop(columns="run_id"))

## Шаг 6. Выбор лучшей модели и регистрация

Критерии в порядке приоритета:
1. **Качество:** `f1` на valid (баланс пропущенного фрода и ложных блокировок), затем `roc_auc`.
2. **Простота и скорость:** при равном качестве (до 4-го знака) выигрывает модель с меньшей глубиной, затем с меньшим числом деревьев — она быстрее в real-time инференсе и проще в интерпретации. Критерий детерминирован, в отличие от замеров времени; время обучения и инференса приведено в таблице справочно.

In [ ]:
runs_df = mlflow.search_runs(experiment_ids=[experiment_id], filter_string="attributes.status = 'FINISHED'")
runs_df["train_time_sec"] = runs_df["tags.train_time_sec"].astype(float)
runs_df["predict_ms_per_1k_rows"] = runs_df["tags.predict_ms_per_1k_rows"].astype(float)
runs_df = (
    runs_df.assign(_f1=runs_df["metrics.f1"].round(4), _auc=runs_df["metrics.roc_auc"].round(4),
                   _depth=runs_df["params.max_depth"].astype(int), _trees=runs_df["params.n_estimators"].astype(int))
    .sort_values(["_f1", "_auc", "_depth", "_trees"], ascending=[False, False, True, True])
    .reset_index(drop=True)
)

METRIC_COLS = ["metrics.accuracy", "metrics.precision", "metrics.recall", "metrics.f1", "metrics.roc_auc"]
view_cols = ["tags.mlflow.runName"] + [f"params.{k}" for k in HP_KEYS] + METRIC_COLS + ["train_time_sec", "predict_ms_per_1k_rows"]
comparison = runs_df[view_cols].rename(columns=lambda c: c.split(".")[-1])
display(comparison.style.format({m.split(".")[-1]: "{:.4f}" for m in METRIC_COLS}))

best = runs_df.iloc[0]
best_run_id, best_run_name = best["run_id"], best["tags.mlflow.runName"]
print(f"Лучший запуск: {best_run_name} ({best_run_id})")

fig, ax = plt.subplots(figsize=(12, 4.5))
comparison.set_index("runName")[["accuracy", "precision", "recall", "f1", "roc_auc"]].plot.bar(ax=ax, width=0.8)
ax.set_ylim(max(0, comparison[["accuracy", "precision", "recall", "f1", "roc_auc"]].min().min() - 0.05), 1.005)
ax.set_title("Сравнение запусков (valid)"); ax.set_xlabel(""); ax.legend(loc="lower right", ncol=5)
ax.tick_params(axis="x", rotation=0)
save_fig(fig, ARTIFACT_DIR / "runs_comparison.png")
client.log_artifact(best_run_id, str(ARTIFACT_DIR / "runs_comparison.png"), artifact_path="comparison")

In [ ]:
model_version = None
try:
    source_uri = best.get("tags.model_uri")
    if not isinstance(source_uri, str):
        source_uri = f"runs:/{best_run_id}/model"
    already = [v for v in client.search_model_versions(f"name='{MODEL_NAME}'") if v.run_id == best_run_id]
    if already:      # повторный запуск тетради: лучший run не изменился → новую версию не плодим
        model_version = max(already, key=lambda v: int(v.version))
    else:
        model_version = mlflow.register_model(source_uri, MODEL_NAME)
    client.set_registered_model_alias(MODEL_NAME, "champion", model_version.version)
    client.update_model_version(MODEL_NAME, model_version.version,
                                description=f"XGBClassifier, run {best_run_name}, valid f1={best['metrics.f1']:.4f}")
    print(f"{MODEL_NAME} v{model_version.version} ({'уже была зарегистрирована' if already else 'новая версия'}), алиас 'champion'")
except Exception as err:
    print("register_model() не сработал (обычно из-за расхождения версий mlflow клиента и сервера):", err)
    print(f"Зарегистрируйте вручную: MLflow UI → эксперимент → run '{best_run_name}' → Register model → '{MODEL_NAME}'")

In [ ]:
# Загрузка из Model Registry (или напрямую из лучшего run) и итоговая проверка на отложенном тесте
model_uri = f"models:/{MODEL_NAME}/{model_version.version}" if model_version else f"runs:/{best_run_id}/model"
prod_model = mlflow.xgboost.load_model(model_uri)
print("Загружена модель:", model_uri)

test_proba = prod_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)
test_metrics = compute_metrics(y_test, test_pred, test_proba)
display(pd.DataFrame({"valid": best[METRIC_COLS].rename(lambda c: c.split(".")[-1]).astype(float),
                      "test": pd.Series(test_metrics)}).round(4))

plot_confusion_matrix(y_test, test_pred, ARTIFACT_DIR / "test_confusion_matrix.png",
                      f"Confusion matrix (test) — {best_run_name}")
client.log_artifact(best_run_id, str(ARTIFACT_DIR / "test_confusion_matrix.png"), artifact_path="test")

# Тестовые метрики — тегами: имена метрик в эксперименте ограничены валидатором
for k, v in test_metrics.items():
    client.set_tag(best_run_id, f"test_{k}", round(v, 4))
    if model_version:
        client.set_model_version_tag(MODEL_NAME, model_version.version, f"test_{k}", round(v, 4))

## Шаг 7. Автопроверка и фиксация окружения

In [ ]:
VALIDATOR = Path("validate_mlflow.py")
if not VALIDATOR.exists():
    urllib.request.urlretrieve("https://code.s3.yandex.net/data-scientist/dsplus/validate_mlflow.py", VALIDATOR)

proc = subprocess.run([sys.executable, str(VALIDATOR), EXPERIMENT_NAME], capture_output=True, text=True)
print(proc.stdout[-3000:], proc.stderr[-1500:] if proc.returncode else "")

In [ ]:
# requirements.txt с фактическими версиями окружения, в котором получен результат
PACKAGES = ["mlflow", "xgboost", "scikit-learn", "shap", "pandas", "numpy", "scipy", "matplotlib",
            "seaborn", "python-dotenv", "pyarrow", "boto3"]
lines = []
for pkg in PACKAGES:
    try:
        lines.append(f"{pkg}=={metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        print("не установлен:", pkg)
Path("requirements.txt").write_text("\n".join(lines) + "\n")
print(f"# python {sys.version.split()[0]}")
print("\n".join(lines))

## Шаг 8. Итоговый вывод

In [ ]:
worst = runs_df.iloc[-1]
ties = int((runs_df["metrics.f1"].round(4) == round(best["metrics.f1"], 4)).sum())
cm = confusion_matrix(y_test, test_pred)
hp = ", ".join(f"`{k}={best[f'params.{k}']}`" for k in HP_KEYS)

display(Markdown(f'''
**Данные.** 100 000 транзакций, пропусков и дубликатов нет, доля фрода {df[TARGET].mean():.1%}. Исключены: утечка `fraud_risk`,
дубликат `unusual_location_flag` (= `is_international`), константный `hour`.

**Гипотезы.** Подтверждены H1 (международные транзакции, V = {hyp_df.loc[0, "cramers_v"]}) и H4 (высокорисковые устройства,
V = {hyp_df.loc[3, "cramers_v"]}). Не подтверждены H2 (метод оплаты, p = {hyp_df.loc[1, "p_value"]:.2f}) и H3 (день недели, p = {hyp_df.loc[2, "p_value"]:.2f}).

**Эксперименты.** В MLflow-эксперименте `{EXPERIMENT_NAME}` завершённых запусков: {len(runs_df)}. Разброс качества на valid:
f1 от {worst["metrics.f1"]:.4f} (`{worst["tags.mlflow.runName"]}`) до {best["metrics.f1"]:.4f} (`{best_run_name}`).
Запусков с лучшим f1 (с точностью до 4 знаков): {ties}; среди равных выбран самый простой (минимальная глубина, затем число деревьев).

**Лучшая модель — `{best_run_name}`**: {hp}.
- valid: accuracy = {best["metrics.accuracy"]:.4f}, precision = {best["metrics.precision"]:.4f}, recall = {best["metrics.recall"]:.4f}, f1 = {best["metrics.f1"]:.4f}, roc_auc = {best["metrics.roc_auc"]:.4f}; время обучения {best["train_time_sec"]:.2f} с.
- **test** (модель загружена из Model Registry): accuracy = {test_metrics["accuracy"]:.4f}, precision = {test_metrics["precision"]:.4f}, recall = {test_metrics["recall"]:.4f}, f1 = {test_metrics["f1"]:.4f}, roc_auc = {test_metrics["roc_auc"]:.4f}.
- На тесте из {cm[1].sum()} мошеннических транзакций пропущено {cm[1, 0]}, из {cm[0].sum()} легитимных ложно заблокировано {cm[0, 1]}.
- Расхождение f1 между valid и test: {abs(best["metrics.f1"] - test_metrics["f1"]):.4f} — {"переобучения нет" if abs(best["metrics.f1"] - test_metrics["f1"]) < 0.005 else "есть признаки переобучения, стоит усилить регуляризацию"}.

**Model Registry.** Модель зарегистрирована как `{MODEL_NAME}`{f" v{model_version.version}, алиас `champion`" if model_version else " (вручную через UI)"}.
При ежемесячном дообучении новая версия сравнивается с `champion` по тем же метрикам; откат — переносом алиаса на предыдущую версию.

**Оговорка.** Датасет синтетический: качество ≈ 1.0 достигается в большинстве запусков, включая модели минимальной глубины (см. таблицу сравнения),
т. е. целевая переменная задана детерминированным правилом — порогом по аддитивному скорингу признаков риска. SHAP это подтверждает: вклад дают
флаги риска и счётчики активности, а страна, категория продавца и метод оплаты — нулевой. На реальных данных такого качества ожидать нельзя; там ключевыми станут подбор порога под стоимость ошибок
(пропущенный фрод vs ложная блокировка) и валидация по времени, а не случайным разбиением.
'''))